# ModeSwitch-LLM Selector Notebook

This notebook handles the selector side of the project.

It does not rerun benchmarks. It only parses the measurements that are already saved in:

- `baseline_smoke_test (3) (1).ipynb`
- `stress_test (2).ipynb`

Then it turns those results into a small routing problem and checks whether a simple selector is actually justified.


## Proposal Alignment

The proposal asks for:

1. static baselines
2. a phase-agnostic comparison baseline
3. a simple phase-aware selector
4. routing based on prompt length, expected output length, memory pressure, and latency target
5. evaluation on TTFT, TBT, memory, energy, and quality

So the notebook is organized around those exact pieces.


In [1]:
from __future__ import annotations

import json
import re
from io import StringIO
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
CONTROLLER_NOTEBOOK = ROOT / "baseline_smoke_test (3) (1).ipynb"
STRESS_NOTEBOOK = ROOT / "stress_test (2).ipynb"
CONFIG_PATH = ROOT / "config.py"

RUN_RE = re.compile(
    r"\[(\d+)/(\d+)\]\s+([A-Za-z0-9_]+)\s+\|\s+([A-Za-z0-9_]+)\s+\|\s+trial=(\d+)\s+\| "
    r"ttft=([\d.]+) ms \| lat=([\d.]+) ms \| tps=([\d.]+) \| J/tok=([\d.]+) \| gpu=([\d.]+) MB"
)
BASE_MODEL_RE = re.compile(r'^\s*model_name_or_path:\s*str\s*=\s*"([^"]+)"', re.M)
TOKENIZER_RE = re.compile(r'^\s*tokenizer_name_or_path:\s*Optional\[str\]\s*=\s*"([^"]+)"', re.M)
SPECIAL_MODEL_RE = re.compile(
    r'^\s*(\w+_model_name_or_path):\s*Optional\[str\]\s*=\s*"([^"]+)"', re.M
)
OUTPUT_TOKENS = {
    "ctx4032_out64": 64,
    "ctx3840_out256": 256,
    "shared4032_out64": 64,
    "ctx2048_out64": 64,
}
STATIC_BASELINE_EXPECTED = {
    "static_fp16": {
        "mean_ttft_ms": 279.397,
        "mean_total_latency_ms": 3716.281,
        "mean_tokens_per_second": 41.498,
        "mean_energy_per_token_j": 4.947,
        "mean_quality_score": 1.0,
    },
    "static_int8": {
        "mean_ttft_ms": 224.595,
        "mean_total_latency_ms": 2009.324,
        "mean_tokens_per_second": 75.931,
        "mean_energy_per_token_j": 2.616,
        "mean_quality_score": 0.83682,
    },
    "static_speculative": {
        "mean_ttft_ms": 353.023,
        "mean_total_latency_ms": 1499.481,
        "mean_tokens_per_second": 98.161,
        "mean_energy_per_token_j": 2.382,
        "mean_quality_score": 1.0,
    },
    "static_gptq": {
        "mean_ttft_ms": 319.298,
        "mean_total_latency_ms": 1625.135,
        "mean_tokens_per_second": 92.843,
        "mean_energy_per_token_j": 2.385,
        "mean_quality_score": 0.228814,
    },
}
QUALITY_RETENTION_FLOOR = 0.85


def parse_logged_runs(path: Path) -> pd.DataFrame:
    nb = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for cell in nb["cells"]:
        for output in cell.get("outputs", []):
            if output.get("output_type") != "stream":
                continue
            text = "".join(output.get("text", []))
            for match in RUN_RE.finditer(text):
                workload_name = match.group(4)
                ttft_ms = float(match.group(6))
                total_latency_ms = float(match.group(7))
                output_tokens = OUTPUT_TOKENS.get(workload_name)
                rows.append(
                    {
                        "mode_name": match.group(3),
                        "workload_name": workload_name,
                        "trial_index": int(match.group(5)),
                        "ttft_ms": ttft_ms,
                        "total_latency_ms": total_latency_ms,
                        "tokens_per_second": float(match.group(8)),
                        "energy_per_token_j": float(match.group(9)),
                        "peak_gpu_memory_mb": float(match.group(10)),
                        "output_tokens": output_tokens,
                        "prefill_ms": ttft_ms,
                        "tbt_ms": ((total_latency_ms - ttft_ms) / output_tokens)
                        if output_tokens
                        else None,
                    }
                )
    return pd.DataFrame(rows)


def aggregate_runs(df: pd.DataFrame, source: str) -> pd.DataFrame:
    metrics = [
        "ttft_ms",
        "total_latency_ms",
        "tokens_per_second",
        "energy_per_token_j",
        "peak_gpu_memory_mb",
        "prefill_ms",
        "tbt_ms",
    ]
    out = df.groupby(["mode_name", "workload_name"])[metrics].agg(["mean", "std"]).reset_index()
    out.columns = ["mode_name", "workload_name"] + [
        f"{metric}_{stat}" for metric, stat in out.columns.tolist()[2:]
    ]
    out["source"] = source
    return out


def load_html_table(path: Path, cell_index: int, output_index: int) -> pd.DataFrame:
    nb = json.loads(path.read_text(encoding="utf-8"))
    output = nb["cells"][cell_index]["outputs"][output_index]
    html = output["data"]["text/html"]
    if isinstance(html, list):
        html = "".join(html)
    return pd.read_html(StringIO(html))[0]


def extract_model_inventory(config_path: Path) -> pd.DataFrame:
    config_text = config_path.read_text(encoding="utf-8")
    base_model = BASE_MODEL_RE.search(config_text).group(1)
    tokenizer = TOKENIZER_RE.search(config_text).group(1)
    rows = [
        {
            "mode_family": "base_fp16",
            "deployed_model": base_model,
            "notes": "Main checkpoint used for fp16_baseline and non-quantized vLLM modes.",
        },
        {
            "mode_family": "tokenizer",
            "deployed_model": tokenizer,
            "notes": "Tokenizer paired with the base checkpoint.",
        },
    ]
    for key, value in SPECIAL_MODEL_RE.findall(config_text):
        rows.append(
            {
                "mode_family": key.replace("_model_name_or_path", ""),
                "deployed_model": value,
                "notes": "Configured special checkpoint or draft model.",
            }
        )
    rows.extend(
        [
            {
                "mode_family": "runtime variants",
                "deployed_model": base_model,
                "notes": "prefix_caching, chunked_prefill, continuous_batching, and cuda_graphs all reuse the base model.",
            },
            {
                "mode_family": "hybrid modes",
                "deployed_model": "compositions of configured single modes",
                "notes": "Measured directly in the long-context stress benchmark.",
            },
        ]
    )
    return pd.DataFrame(rows)


def build_quality_proxy(controller_agg: pd.DataFrame) -> dict[str, float]:
    mode_quality = (
        controller_agg.groupby("mode_name", as_index=False)["quality_score"]
        .median()
        .sort_values("quality_score", ascending=False)
    )
    base_quality = dict(zip(mode_quality["mode_name"], mode_quality["quality_score"]))
    for hybrid_name, components in {
        "int8_plus_chunked_prefill": ["int8_quant", "chunked_prefill"],
        "gptq_plus_chunked_prefill": ["gptq_4bit", "chunked_prefill"],
        "speculative_plus_chunked_prefill": ["speculative_decoding", "chunked_prefill"],
        "int8_plus_prefix_caching": ["int8_quant", "prefix_caching"],
        "gptq_plus_prefix_caching": ["gptq_4bit", "prefix_caching"],
        "int8_plus_continuous_batching": ["int8_quant", "continuous_batching"],
        "gptq_plus_continuous_batching": ["gptq_4bit", "continuous_batching"],
    }.items():
        base_quality[hybrid_name] = min(base_quality[component] for component in components)
    return base_quality


def phase_plan_for_mode(mode_name: str) -> tuple[str, str]:
    if mode_name == "fp16_baseline":
        return ("fp16 prefill", "fp16 decode")
    if mode_name == "int8_quant":
        return ("int8 prefill", "int8 decode")
    if mode_name == "gptq_4bit":
        return ("gptq prefill", "gptq decode")
    if mode_name == "speculative_decoding":
        return ("standard prefill", "speculative decode")
    if mode_name == "prefix_caching":
        return ("prefix-cache prefill", "standard decode")
    if mode_name == "int8_plus_prefix_caching":
        return ("prefix-cache prefill", "int8 decode")
    if mode_name == "gptq_plus_prefix_caching":
        return ("prefix-cache prefill", "gptq decode")
    if mode_name == "int8_plus_continuous_batching":
        return ("batched int8 prefill", "batched int8 decode")
    if mode_name == "gptq_plus_continuous_batching":
        return ("batched gptq prefill", "batched gptq decode")
    if mode_name == "chunked_prefill":
        return ("chunked prefill", "standard decode")
    return (mode_name, mode_name)


def route(selector_name: str, signals_row: pd.Series) -> str:
    if selector_name == "static_fp16":
        return "fp16_baseline"
    if selector_name == "static_int8":
        return "int8_quant"
    if selector_name == "static_speculative":
        return "speculative_decoding"
    if selector_name == "static_gptq":
        return "gptq_4bit"

    if selector_name == "phase_agnostic_simple":
        if signals_row["latency_target"] == "throughput_serving":
            return "int8_plus_continuous_batching"
        if signals_row["repeated_prefix"]:
            return "prefix_caching"
        if signals_row["predicted_output_length"] <= 64:
            return "speculative_decoding"
        return "int8_quant"

    if selector_name == "phase_aware_quality":
        if signals_row["latency_target"] == "throughput_serving":
            return "int8_plus_continuous_batching"
        if signals_row["repeated_prefix"]:
            return (
                "int8_plus_prefix_caching"
                if signals_row["memory_pressure"] >= 0.9
                else "prefix_caching"
            )
        if (
            signals_row["predicted_output_length"] <= 64
            and signals_row["latency_target"] == "interactive"
            and signals_row["memory_pressure"] < 0.98
        ):
            return "speculative_decoding"
        return "int8_quant"

    if selector_name == "phase_aware_aggressive":
        if signals_row["latency_target"] == "throughput_serving":
            return "int8_plus_continuous_batching"
        if signals_row["repeated_prefix"]:
            return "gptq_plus_prefix_caching"
        if signals_row["predicted_output_length"] <= 64:
            return "gptq_4bit"
        return "speculative_decoding"

    raise KeyError(f"Unknown selector: {selector_name}")


def build_selector_routes(workload_signals_df: pd.DataFrame, selector_names: list[str]) -> pd.DataFrame:
    rows = []
    for selector_name in selector_names:
        for _, signal_row in workload_signals_df.iterrows():
            chosen_mode = route(selector_name, signal_row)
            prefill_route, decode_route = phase_plan_for_mode(chosen_mode)
            rows.append(
                {
                    **signal_row.to_dict(),
                    "selector": selector_name,
                    "chosen_mode": chosen_mode,
                    "prefill_route": prefill_route,
                    "decode_route": decode_route,
                }
            )
    return pd.DataFrame(rows)


def summarize_selector_rows(df: pd.DataFrame) -> pd.DataFrame:
    usable = df[df["measured"]].copy()
    summary = (
        usable.groupby("selector", as_index=False)
        .agg(
            workloads_covered=("workload_name", "nunique"),
            mean_ttft_ms=("ttft_ms", "mean"),
            mean_prefill_ms=("prefill_ms", "mean"),
            mean_tbt_ms=("tbt_ms", "mean"),
            mean_total_latency_ms=("total_latency_ms", "mean"),
            mean_tokens_per_second=("tokens_per_second", "mean"),
            mean_energy_per_token_j=("energy_per_token_j", "mean"),
            mean_peak_gpu_memory_mb=("peak_gpu_memory_mb", "mean"),
            mean_quality_score=("quality_score", "mean"),
            mean_ttft_ms_std=("ttft_ms_std", "mean"),
            mean_prefill_ms_std=("prefill_ms_std", "mean"),
            mean_tbt_ms_std=("tbt_ms_std", "mean"),
            mean_total_latency_ms_std=("total_latency_ms_std", "mean"),
            mean_tokens_per_second_std=("tokens_per_second_std", "mean"),
            mean_energy_per_token_j_std=("energy_per_token_j_std", "mean"),
            mean_peak_gpu_memory_mb_std=("peak_gpu_memory_mb_std", "mean"),
        )
        .sort_values("mean_total_latency_ms")
        .reset_index(drop=True)
    )
    return summary


def apply_quality_reference(summary_df: pd.DataFrame) -> pd.DataFrame:
    summary_df = summary_df.copy()
    fp16_row = summary_df.loc[summary_df["selector"] == "static_fp16"]
    fp16_quality = float(fp16_row["mean_quality_score"].iloc[0])
    summary_df["quality_retention_vs_fp16"] = summary_df["mean_quality_score"] / fp16_quality
    summary_df["passes_quality_retention_rule"] = (
        summary_df["quality_retention_vs_fp16"] >= QUALITY_RETENTION_FLOOR
    )
    return summary_df


def sanity_check_static_baselines(summary_df: pd.DataFrame, tolerance: float = 0.02) -> None:
    for selector_name, expected_values in STATIC_BASELINE_EXPECTED.items():
        row = summary_df.loc[summary_df["selector"] == selector_name]
        if row.empty:
            raise AssertionError(f"Missing selector row for {selector_name}.")
        row = row.iloc[0]
        for column_name, expected_value in expected_values.items():
            actual_value = float(row[column_name])
            if abs(actual_value - expected_value) > tolerance:
                raise AssertionError(
                    f"{selector_name} {column_name} mismatch: got {actual_value}, expected {expected_value}"
                )


In [2]:
controller_runs = parse_logged_runs(CONTROLLER_NOTEBOOK)
stress_runs = parse_logged_runs(STRESS_NOTEBOOK)

controller_agg = aggregate_runs(controller_runs, "controller_benchmark")
stress_agg = aggregate_runs(stress_runs, "stress_benchmark")

controller_agg_html = load_html_table(CONTROLLER_NOTEBOOK, 4, 1).drop(
    columns=["Unnamed: 0", "..."], errors="ignore"
)
controller_cmp = load_html_table(CONTROLLER_NOTEBOOK, 4, 3).drop(
    columns=["Unnamed: 0", "..."], errors="ignore"
)

controller_quality = controller_agg_html[
    ["mode_name", "workload_name", "baseline_similarity_rouge_l_f1_mean"]
].rename(columns={"baseline_similarity_rouge_l_f1_mean": "quality_score"})
controller_agg = controller_agg.merge(
    controller_quality, on=["mode_name", "workload_name"], how="left"
)

quality_proxy = build_quality_proxy(controller_agg)
stress_agg["quality_score"] = stress_agg["mode_name"].map(quality_proxy)

all_agg = pd.concat([controller_agg, stress_agg], ignore_index=True, sort=False)

print(f"Controller runs parsed: {len(controller_runs)}")
print(f"Stress and hybrid runs parsed: {len(stress_runs)}")
print(f"Controller mode-workload pairs: {controller_agg.shape[0]}")
print(f"Stress mode-workload pairs: {stress_agg.shape[0]}")


Controller runs parsed: 84
Stress and hybrid runs parsed: 280
Controller mode-workload pairs: 42
Stress mode-workload pairs: 22


C:\Users\hivan\AppData\Local\Temp\ipykernel_36316\4285134728.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_agg = pd.concat([controller_agg, stress_agg], ignore_index=True, sort=False)


## Data Sources

The selector study uses two measurement sources and one interpretation source:

- the controller sweep for broad fixed-mode behavior
- the long-context stress notebook for long-context and hybrid behavior
- the plot/report material for controller hints only


In [3]:
benchmark_inventory_df = pd.DataFrame(
    [
        {
            "artifact": "baseline_smoke_test (3) (1).ipynb",
            "role": "broad controller sweep",
            "parsed_runs": int(len(controller_runs)),
            "mode_workload_pairs": int(controller_agg.shape[0]),
            "what_it_adds": "Fixed-mode behavior across short, long, memory-pressure, and shared-prefix cases.",
        },
        {
            "artifact": "stress_test (2).ipynb",
            "role": "long-context and hybrid sweep",
            "parsed_runs": int(len(stress_runs)),
            "mode_workload_pairs": int(stress_agg.shape[0]),
            "what_it_adds": "Measured long-context hybrids like prefix caching, chunked prefill, and continuous batching.",
        },
        {
            "artifact": "stress_test_plots.ipynb",
            "role": "interpretation only",
            "parsed_runs": 0,
            "mode_workload_pairs": 0,
            "what_it_adds": "Summarizes what the stress results mean for controller logic.",
        },
    ]
)
benchmark_inventory_df


,artifact,role,parsed_runs,mode_workload_pairs,what_it_adds
0,baseline_smoke_test (3) (1).ipynb,broad controller sweep,84,42,"Fixed-mode behavior across short, long, memory..."
1,stress_test (2).ipynb,long-context and hybrid sweep,280,22,Measured long-context hybrids like prefix cach...
2,stress_test_plots.ipynb,interpretation only,0,0,Summarizes what the stress results mean for co...


In [4]:
model_inventory_df = extract_model_inventory(CONFIG_PATH)
model_inventory_df


,mode_family,deployed_model,notes
0,base_fp16,meta-llama/Meta-Llama-3.1-8B-Instruct,Main checkpoint used for fp16_baseline and non...
1,tokenizer,meta-llama/Meta-Llama-3.1-8B-Instruct,Tokenizer paired with the base checkpoint.
2,int8,RedHatAI/Meta-Llama-3.1-8B-Instruct-quantized....,Configured special checkpoint or draft model.
3,awq,hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-...,Configured special checkpoint or draft model.
4,gptq,hugging-quants/Meta-Llama-3.1-8B-Instruct-GPTQ...,Configured special checkpoint or draft model.
5,speculative,meta-llama/Llama-3.2-1B-Instruct,Configured special checkpoint or draft model.
6,runtime variants,meta-llama/Meta-Llama-3.1-8B-Instruct,"prefix_caching, chunked_prefill, continuous_ba..."
7,hybrid modes,compositions of configured single modes,Measured directly in the long-context stress b...


## Fixed-Mode Baselines

Before building selector logic, it helps to know which single modes are even worth reasoning over.

The next tables come from the controller sweep:

- a compact fixed-mode ranking
- the quality-safe winner per workload using a relative rule: retain at least 85% of the FP16 quality score on that workload


In [5]:
controller_mode_summary = (
    controller_cmp.groupby("mode_name", as_index=False)
    .agg(
        workloads_measured=("workload_name", "count"),
        mean_latency_speedup=("latency_speedup_vs_baseline", "mean"),
        mean_throughput_ratio=("throughput_ratio_vs_baseline", "mean"),
        mean_energy_ratio=("energy_per_token_ratio_vs_baseline", "mean"),
        median_quality=("baseline_similarity_rouge_l_f1_mean", "median"),
    )
    .sort_values(["median_quality", "mean_latency_speedup"], ascending=[False, False])
    .reset_index(drop=True)
)
controller_mode_summary["takeaway"] = [
    "strong quality-safe decode specialist" if name == "speculative_decoding"
    else "best general fixed-mode baseline" if name == "int8_quant"
    else "strong repeated-prefix fixed baseline" if name == "prefix_caching"
    else "useful phase primitive more than a whole-request default" if name == "chunked_prefill"
    else "better as a serving extension than a default interactive branch" if name == "continuous_batching"
    else "quality-safe but not strong enough to prioritize first" if name == "cuda_graphs"
    else "not strong enough for the first selector version" if name == "kv_cache_compression"
    else "fast but quality-limited" if ("gptq" in name or "awq" in name)
    else "measured fixed mode"
    for name in controller_mode_summary["mode_name"]
]
controller_mode_summary


,mode_name,workloads_measured,mean_latency_speedup,mean_throughput_ratio,mean_energy_ratio,median_quality,takeaway
0,speculative_decoding,3,2.069220,2.069201,0.442784,1.000000,strong quality-safe decode specialist
1,cuda_graphs,3,1.406624,1.406551,0.850473,1.000000,quality-safe but not strong enough to prioriti...
2,prefix_caching,2,1.387617,1.387549,0.849903,1.000000,strong repeated-prefix fixed baseline
3,chunked_prefill,3,1.383936,1.383990,0.852012,1.000000,useful phase primitive more than a whole-reque...
4,int8_quant,5,1.876532,1.876390,0.478920,0.836820,best general fixed-mode baseline
5,kv_cache_compression,3,1.085325,1.085313,1.015603,0.700000,not strong enough for the first selector version
6,awq_4bit,5,1.792220,1.792175,0.599657,0.329412,fast but quality-limited
7,continuous_batching,3,1.175476,4.772568,0.276280,0.287293,better as a serving extension than a default i...
8,gptq_4bit,5,2.567178,2.588189,0.380089,0.228814,fast but quality-limited
9,awq_plus_chunked_prefill,3,1.741027,1.741082,0.626181,0.210084,fast but quality-limited


In [6]:
controller_fp16_quality = controller_cmp[controller_cmp["mode_name"] == "fp16_baseline"][
    ["workload_name", "baseline_similarity_rouge_l_f1_mean"]
].rename(columns={"baseline_similarity_rouge_l_f1_mean": "fp16_quality_score"})

controller_cmp_with_ref = controller_cmp.merge(
    controller_fp16_quality, on="workload_name", how="left"
)
controller_cmp_with_ref["quality_retention_vs_fp16"] = (
    controller_cmp_with_ref["baseline_similarity_rouge_l_f1_mean"]
    / controller_cmp_with_ref["fp16_quality_score"]
)

quality_safe_best = (
    controller_cmp_with_ref[
        controller_cmp_with_ref["quality_retention_vs_fp16"] >= QUALITY_RETENTION_FLOOR
    ]
    .groupby("workload_name")["latency_speedup_vs_baseline"]
    .idxmax()
)
quality_safe_best = (
    controller_cmp_with_ref.loc[
        quality_safe_best,
        [
            "workload_name",
            "mode_name",
            "latency_speedup_vs_baseline",
            "energy_per_token_ratio_vs_baseline",
            "baseline_similarity_rouge_l_f1_mean",
            "quality_retention_vs_fp16",
        ],
    ]
    .sort_values("workload_name")
    .reset_index(drop=True)
    .rename(
        columns={
            "mode_name": "quality_safe_winner",
            "latency_speedup_vs_baseline": "latency_speedup_vs_fp16",
            "energy_per_token_ratio_vs_baseline": "energy_ratio_vs_fp16",
            "baseline_similarity_rouge_l_f1_mean": "quality_score",
        }
    )
)
quality_safe_best


,workload_name,quality_safe_winner,latency_speedup_vs_fp16,energy_ratio_vs_fp16,quality_score,quality_retention_vs_fp16


## Runtime Signals

The proposal wanted a simple controller, not a learned policy.

So the routing logic uses:

- prompt length
- predicted output length
- repeated-prefix structure
- memory pressure
- latency target

`memory_pressure` and `latency_target` are both active:

- `memory_pressure` decides whether repeated-prefix traffic stays on `prefix_caching` or moves to `int8_plus_prefix_caching`
- `latency_target` is what moves the serving-style row toward `int8_plus_continuous_batching`


In [7]:
workload_signals_df = pd.DataFrame(
    [
        {
            "workload_name": "ctx4032_out64",
            "prompt_length": 4032,
            "predicted_output_length": 64,
            "repeated_prefix": False,
            "memory_pressure": 0.95,
            "latency_target": "interactive",
            "quality_priority": "high",
        },
        {
            "workload_name": "ctx3840_out256",
            "prompt_length": 3840,
            "predicted_output_length": 256,
            "repeated_prefix": False,
            "memory_pressure": 0.95,
            "latency_target": "balanced",
            "quality_priority": "high",
        },
        {
            "workload_name": "shared4032_out64",
            "prompt_length": 3936,
            "predicted_output_length": 64,
            "repeated_prefix": True,
            "memory_pressure": 0.92,
            "latency_target": "interactive",
            "quality_priority": "high",
        },
        {
            "workload_name": "ctx2048_out64",
            "prompt_length": 2048,
            "predicted_output_length": 64,
            "repeated_prefix": False,
            "memory_pressure": 0.55,
            "latency_target": "throughput_serving",
            "quality_priority": "medium",
        },
    ]
)
workload_signals_df


,workload_name,prompt_length,predicted_output_length,repeated_prefix,memory_pressure,latency_target,quality_priority
0,ctx4032_out64,4032,64,False,0.95,interactive,high
1,ctx3840_out256,3840,256,False,0.95,balanced,high
2,shared4032_out64,3936,64,True,0.92,interactive,high
3,ctx2048_out64,2048,64,False,0.55,throughput_serving,medium


## Measured Candidate Table

This is the actual measured candidate pool for the selector comparison.

Two metric definitions matter here:

- `prefill_ms = ttft_ms`
- `tbt_ms = (total_latency_ms - ttft_ms) / output_tokens`

That second metric matters because the project is built around the prefill vs decode split.

For hybrid quality, the notebook keeps the conservative proxy:

- hybrid quality = minimum quality of the component single modes


In [8]:
selector_cases = (
    all_agg[all_agg["workload_name"].isin(["ctx4032_out64", "ctx3840_out256", "shared4032_out64"])]
    .dropna(subset=["quality_score"])
    .copy()
)
selector_cases = selector_cases.rename(
    columns={
        "ttft_ms_mean": "ttft_ms",
        "ttft_ms_std": "ttft_ms_std",
        "prefill_ms_mean": "prefill_ms",
        "prefill_ms_std": "prefill_ms_std",
        "tbt_ms_mean": "tbt_ms",
        "tbt_ms_std": "tbt_ms_std",
        "total_latency_ms_mean": "total_latency_ms",
        "total_latency_ms_std": "total_latency_ms_std",
        "tokens_per_second_mean": "tokens_per_second",
        "tokens_per_second_std": "tokens_per_second_std",
        "energy_per_token_j_mean": "energy_per_token_j",
        "energy_per_token_j_std": "energy_per_token_j_std",
        "peak_gpu_memory_mb_mean": "peak_gpu_memory_mb",
        "peak_gpu_memory_mb_std": "peak_gpu_memory_mb_std",
    }
)
measured_candidate_df = selector_cases[
    [
        "workload_name",
        "mode_name",
        "source",
        "ttft_ms",
        "ttft_ms_std",
        "prefill_ms",
        "prefill_ms_std",
        "tbt_ms",
        "tbt_ms_std",
        "total_latency_ms",
        "total_latency_ms_std",
        "tokens_per_second",
        "tokens_per_second_std",
        "energy_per_token_j",
        "energy_per_token_j_std",
        "peak_gpu_memory_mb",
        "peak_gpu_memory_mb_std",
        "quality_score",
    ]
].sort_values(["workload_name", "total_latency_ms", "mode_name"]).reset_index(drop=True)
measured_candidate_df


,workload_name,mode_name,source,ttft_ms,ttft_ms_std,prefill_ms,prefill_ms_std,tbt_ms,tbt_ms_std,total_latency_ms,total_latency_ms_std,tokens_per_second,tokens_per_second_std,energy_per_token_j,energy_per_token_j_std,peak_gpu_memory_mb,peak_gpu_memory_mb_std,quality_score
0,ctx3840_out256,speculative_decoding,stress_benchmark,347.581333,9.550378,347.581333,9.550378,7.222451,0.072412,2196.528667,20.716298,116.556667,1.094053,1.797000,0.039343,33519.222000,67.847537,1.000000
1,ctx3840_out256,speculative_plus_chunked_prefill,stress_benchmark,352.573000,7.509308,352.573000,7.509308,7.232020,0.043123,2203.970000,8.169383,116.157000,0.430169,1.799100,0.057949,33587.210000,0.000000,1.000000
2,ctx3840_out256,gptq_plus_chunked_prefill,stress_benchmark,317.225000,12.702125,317.225000,12.702125,7.413070,0.025060,2214.971000,14.015577,115.581000,0.732537,1.825600,0.063488,33625.940000,0.000000,0.228814
3,ctx3840_out256,gptq_4bit,stress_benchmark,315.683333,11.552125,315.683333,11.552125,8.385721,0.642486,2462.428000,166.729776,104.438000,7.516554,1.957200,0.086146,33524.150000,86.965037,0.228814
4,ctx3840_out256,int8_quant,stress_benchmark,219.574000,9.283454,219.574000,9.283454,11.217328,0.277149,3091.210000,69.758146,82.850667,1.742566,2.326200,0.083829,33498.857333,104.733160,0.836820
5,ctx3840_out256,int8_plus_chunked_prefill,stress_benchmark,226.572000,8.346772,226.572000,8.346772,11.205863,0.033579,3095.273000,15.967436,82.709000,0.424642,2.292200,0.030254,33564.320000,0.000000,0.836820
6,ctx3840_out256,chunked_prefill,stress_benchmark,308.610667,11.370368,308.610667,11.370368,14.937635,0.155267,4132.645333,43.296335,61.952667,0.634625,3.789867,0.026454,33523.858000,48.424908,1.000000
7,ctx3840_out256,fp16_baseline,stress_benchmark,276.738667,0.960364,276.738667,0.960364,21.488206,0.267570,5777.719333,67.863314,44.314667,0.515861,4.312800,0.066962,33466.681333,139.891644,1.000000
8,ctx4032_out64,gptq_4bit,stress_benchmark,322.912667,10.884323,322.912667,10.884323,7.264531,0.049179,787.842667,10.676862,81.248667,1.092473,2.813400,0.161639,33536.698000,59.247398,0.228814
9,ctx4032_out64,gptq_plus_chunked_prefill,stress_benchmark,322.991000,9.471828,322.991000,9.471828,7.269141,0.035410,788.216000,10.056532,81.208000,1.044326,2.833900,0.125562,33640.950000,0.000000,0.228814


## Workload-Level Measured Winners

This table is here to make the routing less arbitrary.

For each measured workload, it shows:

- the raw latency winner
- the best prefill
- the best token-by-token decode cost
- the best quality-safe latency winner
- the best quality-safe energy winner

The selector rules below are meant to follow these patterns, not guess around them.


In [9]:
def winner_row(df: pd.DataFrame, column_name: str) -> pd.Series:
    return df.loc[df[column_name].idxmin()]


def workload_quality_reference(group: pd.DataFrame) -> float:
    fp16 = group.loc[group["mode_name"] == "fp16_baseline", "quality_score"]
    if not fp16.empty and pd.notna(fp16.iloc[0]):
        return float(fp16.iloc[0])
    return float(group["quality_score"].max())


evidence_rows = []
for workload_name, group in measured_candidate_df.groupby("workload_name"):
    quality_reference = workload_quality_reference(group)
    quality_safe = group[group["quality_score"] >= quality_reference * QUALITY_RETENTION_FLOOR].copy()
    raw_latency = winner_row(group, "total_latency_ms")
    raw_prefill = winner_row(group, "prefill_ms")
    raw_tbt = winner_row(group, "tbt_ms")
    safe_latency = winner_row(quality_safe, "total_latency_ms") if not quality_safe.empty else None
    safe_energy = winner_row(quality_safe, "energy_per_token_j") if not quality_safe.empty else None
    evidence_rows.append(
        {
            "workload_name": workload_name,
            "quality_reference_score": quality_reference,
            "quality_floor": quality_reference * QUALITY_RETENTION_FLOOR,
            "raw_latency_winner": raw_latency["mode_name"],
            "best_prefill_mode": raw_prefill["mode_name"],
            "best_tbt_mode": raw_tbt["mode_name"],
            "quality_safe_latency_winner": safe_latency["mode_name"] if safe_latency is not None else None,
            "quality_safe_energy_winner": safe_energy["mode_name"] if safe_energy is not None else None,
        }
    )

workload_evidence_df = pd.DataFrame(evidence_rows).sort_values("workload_name").reset_index(drop=True)
workload_evidence_df


,workload_name,quality_reference_score,quality_floor,raw_latency_winner,best_prefill_mode,best_tbt_mode,quality_safe_latency_winner,quality_safe_energy_winner
0,ctx3840_out256,1.0,0.85,speculative_decoding,int8_quant,speculative_decoding,speculative_decoding,speculative_decoding
1,ctx4032_out64,1.0,0.85,gptq_4bit,int8_quant,speculative_decoding,speculative_decoding,speculative_plus_chunked_prefill
2,shared4032_out64,1.0,0.85,gptq_plus_prefix_caching,int8_plus_prefix_caching,gptq_plus_prefix_caching,prefix_caching,prefix_caching


## Selector Variants

The notebook compares four static baselines and three adaptive policies:

- `static_fp16`
- `static_int8`
- `static_speculative`
- `static_gptq`
- `phase_agnostic_simple`
- `phase_aware_quality`
- `phase_aware_aggressive`

The important difference is that the phase-aware selectors make the prefill and decode plan explicit for the chosen route.
The quality guard is also relative to FP16 rather than a hard absolute cutoff.


In [10]:
selector_policy_df = pd.DataFrame(
    [
        {
            "selector": "phase_agnostic_simple",
            "prefill_decision": "No separate prefill branch. Whole-request mode is fixed by workload signals.",
            "decode_decision": "No separate decode branch. Repeated-prefix goes to prefix_caching, short-output interactive goes to speculative, long generation goes to int8.",
            "benchmark_anchor": "Matches quality-safe single-mode winners but intentionally ignores the measured repeated-prefix hybrid gain.",
        },
        {
            "selector": "phase_aware_quality",
            "prefill_decision": "Repeated-prefix traffic under high memory pressure switches to prefix-cache-aware prefill.",
            "decode_decision": "Short-output interactive keeps speculative decode, long generation uses int8 decode, repeated-prefix uses int8 decode after cached-prefix prefill.",
            "benchmark_anchor": "Follows the quality-safe measured pattern in workload_evidence_df and measured_candidate_df.",
        },
        {
            "selector": "phase_aware_aggressive",
            "prefill_decision": "Repeated-prefix traffic still uses prefix-cache-aware prefill, but the quality-safe guard is removed.",
            "decode_decision": "Short-output interactive uses gptq decode, long generation keeps speculative, repeated-prefix uses gptq decode after cached-prefix prefill.",
            "benchmark_anchor": "Tracks raw-latency or low-energy winners even when the relative-to-FP16 quality retention rule is not met.",
        },
    ]
)
selector_policy_df


,selector,prefill_decision,decode_decision,benchmark_anchor
0,phase_agnostic_simple,No separate prefill branch. Whole-request mode...,No separate decode branch. Repeated-prefix goe...,Matches quality-safe single-mode winners but i...
1,phase_aware_quality,Repeated-prefix traffic under high memory pres...,Short-output interactive keeps speculative dec...,Follows the quality-safe measured pattern in w...
2,phase_aware_aggressive,Repeated-prefix traffic still uses prefix-cach...,"Short-output interactive uses gptq decode, lon...",Tracks raw-latency or low-energy winners even ...


In [11]:
selector_names = [
    "static_fp16",
    "static_int8",
    "static_speculative",
    "static_gptq",
    "phase_agnostic_simple",
    "phase_aware_quality",
    "phase_aware_aggressive",
]

core_selector_routes_df = build_selector_routes(
    workload_signals_df[workload_signals_df["workload_name"] != "ctx2048_out64"],
    selector_names,
)

routing_breakdown_df = core_selector_routes_df.merge(
    measured_candidate_df,
    left_on=["workload_name", "chosen_mode"],
    right_on=["workload_name", "mode_name"],
    how="left",
)
routing_breakdown_df["measured"] = routing_breakdown_df["total_latency_ms"].notna()
routing_breakdown_df = routing_breakdown_df.drop(columns=["mode_name"])

selector_summary_full_df = apply_quality_reference(summarize_selector_rows(routing_breakdown_df))

common_subset_workloads = sorted(
    workload_name
    for workload_name, group in routing_breakdown_df.groupby("workload_name")
    if group["measured"].all()
)
selector_summary_common_subset_df = apply_quality_reference(
    summarize_selector_rows(
        routing_breakdown_df[routing_breakdown_df["workload_name"].isin(common_subset_workloads)]
    )
)

sanity_check_static_baselines(selector_summary_full_df)

selector_summary_full_df["comparison_scope"] = "full"
selector_summary_common_subset_df["comparison_scope"] = "common_subset"

print("Static baseline sanity checks passed.")
print("Common-subset workloads:", ", ".join(common_subset_workloads))


Static baseline sanity checks passed.
Common-subset workloads: ctx3840_out256, ctx4032_out64


## Per-Workload Routing Breakdown

This is the table that makes the selector behavior visible.

If the repeated-prefix branch matters, it should show up directly here and not only inside an average.


In [12]:
routing_breakdown_df = routing_breakdown_df[
    [
        "workload_name",
        "selector",
        "chosen_mode",
        "prefill_route",
        "decode_route",
        "prefill_ms",
        "tbt_ms",
        "total_latency_ms",
        "energy_per_token_j",
        "peak_gpu_memory_mb",
        "quality_score",
        "measured",
    ]
].sort_values(["selector", "workload_name"]).reset_index(drop=True)
routing_breakdown_df


,workload_name,selector,chosen_mode,prefill_route,decode_route,prefill_ms,tbt_ms,total_latency_ms,energy_per_token_j,peak_gpu_memory_mb,quality_score,measured
0,ctx3840_out256,phase_agnostic_simple,int8_quant,int8 prefill,int8 decode,219.574000,11.217328,3091.210000,2.326200,33498.857333,0.836820,True
1,ctx4032_out64,phase_agnostic_simple,speculative_decoding,standard prefill,speculative decode,358.464667,6.937000,802.432667,2.967467,33515.101333,1.000000,True
2,shared4032_out64,phase_agnostic_simple,prefix_caching,prefix-cache prefill,standard decode,52.176000,14.957740,1009.471333,3.821133,33544.347333,1.000000,True
3,ctx3840_out256,phase_aware_aggressive,speculative_decoding,standard prefill,speculative decode,347.581333,7.222451,2196.528667,1.797000,33519.222000,1.000000,True
4,ctx4032_out64,phase_aware_aggressive,gptq_4bit,gptq prefill,gptq decode,322.912667,7.264531,787.842667,2.813400,33536.698000,0.228814,True
5,shared4032_out64,phase_aware_aggressive,gptq_plus_prefix_caching,prefix-cache prefill,gptq decode,54.320000,7.609984,541.359000,2.049400,33634.140000,0.228814,True
6,ctx3840_out256,phase_aware_quality,int8_quant,int8 prefill,int8 decode,219.574000,11.217328,3091.210000,2.326200,33498.857333,0.836820,True
7,ctx4032_out64,phase_aware_quality,speculative_decoding,standard prefill,speculative decode,358.464667,6.937000,802.432667,2.967467,33515.101333,1.000000,True
8,shared4032_out64,phase_aware_quality,int8_plus_prefix_caching,prefix-cache prefill,int8 decode,49.613000,11.294531,772.463000,2.425600,33574.510000,0.836820,True
9,ctx3840_out256,static_fp16,fp16_baseline,fp16 prefill,fp16 decode,276.738667,21.488206,5777.719333,4.312800,33466.681333,1.000000,True


## Selector Summary Tables

I keep two versions of the summary on purpose:

- `selector_summary_common_subset_df` is the fair apples-to-apples comparison and should be read first
- `selector_summary_full_df` keeps the repeated-prefix row, which is exactly where the phase-aware branch shows up

The pass rule is relative to the FP16 reference:

- a selector passes only if its mean quality retains at least 85% of FP16's mean quality on the same summary scope


In [13]:
selector_summary_common_subset_df


,selector,workloads_covered,mean_ttft_ms,mean_prefill_ms,mean_tbt_ms,mean_total_latency_ms,mean_tokens_per_second,mean_energy_per_token_j,mean_peak_gpu_memory_mb,mean_quality_score,mean_ttft_ms_std,mean_prefill_ms_std,mean_tbt_ms_std,mean_total_latency_ms_std,mean_tokens_per_second_std,mean_energy_per_token_j_std,mean_peak_gpu_memory_mb_std,quality_retention_vs_fp16,passes_quality_retention_rule,comparison_scope
0,phase_aware_aggressive,2,335.247000,335.247000,7.243491,1492.185667,98.902667,2.305200,33527.960000,0.614407,10.217350,10.217350,0.060795,15.696580,1.093263,0.100491,63.547468,0.614407,False,common_subset
1,static_speculative,2,353.023000,353.023000,7.079725,1499.480667,98.161333,2.382233,33517.161667,1.000000,8.670471,8.670471,0.072826,14.885154,0.997545,0.092205,76.571626,1.000000,True,common_subset
2,static_gptq,2,319.298000,319.298000,7.825126,1625.135333,92.843333,2.385300,33530.424000,0.228814,11.218224,11.218224,0.345832,88.703319,4.304514,0.123893,73.106218,0.228814,False,common_subset
3,phase_agnostic_simple,2,289.019333,289.019333,9.077164,1946.821333,81.308333,2.646833,33506.979333,0.918410,8.537009,8.537009,0.175194,39.406079,1.321802,0.114448,95.014437,0.918410,True,common_subset
4,phase_aware_quality,2,289.019333,289.019333,9.077164,1946.821333,81.308333,2.646833,33506.979333,0.918410,8.537009,8.537009,0.175194,39.406079,1.321802,0.114448,95.014437,0.918410,True,common_subset
5,static_int8,2,224.595333,224.595333,11.060393,2009.324000,75.930667,2.616233,33525.175667,0.836820,8.722249,8.722249,0.181444,38.470424,1.139980,0.066878,73.634785,0.836820,False,common_subset
6,static_fp16,2,279.397333,279.397333,21.469004,3716.281333,41.498000,4.946700,33497.518333,1.000000,0.834854,0.834854,0.320455,45.932541,0.536952,0.131829,111.224035,1.000000,True,common_subset


In [14]:
selector_summary_full_df


,selector,workloads_covered,mean_ttft_ms,mean_prefill_ms,mean_tbt_ms,mean_total_latency_ms,mean_tokens_per_second,mean_energy_per_token_j,mean_peak_gpu_memory_mb,mean_quality_score,mean_ttft_ms_std,mean_prefill_ms_std,mean_tbt_ms_std,mean_total_latency_ms_std,mean_tokens_per_second_std,mean_energy_per_token_j_std,mean_peak_gpu_memory_mb_std,quality_retention_vs_fp16,passes_quality_retention_rule,comparison_scope
0,phase_aware_aggressive,3,241.604667,241.604667,7.365655,1175.243444,105.345111,2.219933,33563.353333,0.485876,7.037218,7.037218,0.065008,12.197214,1.103255,0.117723,42.364978,0.485876,False,full
1,static_speculative,2,353.023000,353.023000,7.079725,1499.480667,98.161333,2.382233,33517.161667,1.000000,8.670471,8.670471,0.072826,14.885154,0.997545,0.092205,76.571626,1.000000,True,full
2,phase_aware_quality,3,209.217222,209.217222,9.816286,1555.368556,81.823222,2.573089,33529.489556,0.891213,5.854556,5.854556,0.131073,27.302149,0.991812,0.111602,63.342958,0.891213,True,full
3,static_gptq,2,319.298000,319.298000,7.825126,1625.135333,92.843333,2.385300,33530.424000,0.228814,11.218224,11.218224,0.345832,88.703319,4.304514,0.123893,73.106218,0.228814,False,full
4,phase_agnostic_simple,3,210.071556,210.071556,11.037356,1634.371333,75.340444,3.038267,33519.435333,0.945607,6.091842,6.091842,0.163264,29.430093,1.075112,0.128898,70.542970,0.945607,True,full
5,static_int8,2,224.595333,224.595333,11.060393,2009.324000,75.930667,2.616233,33525.175667,0.836820,8.722249,8.722249,0.181444,38.470424,1.139980,0.066878,73.634785,0.836820,False,full
6,static_fp16,2,279.397333,279.397333,21.469004,3716.281333,41.498000,4.946700,33497.518333,1.000000,0.834854,0.834854,0.320455,45.932541,0.536952,0.131829,111.224035,1.000000,True,full


## Claim Trace Table

Instead of printing a prewritten interpretation paragraph, the notebook builds a trace table.

Each row is a claim worth making in the writeup, plus the exact value and table it came from.


In [15]:
def lookup(summary_df: pd.DataFrame, selector_name: str) -> pd.Series:
    return summary_df.loc[summary_df["selector"] == selector_name].iloc[0]


common_quality = lookup(selector_summary_common_subset_df, "phase_aware_quality")
common_simple = lookup(selector_summary_common_subset_df, "phase_agnostic_simple")
full_quality = lookup(selector_summary_full_df, "phase_aware_quality")
full_aggressive = lookup(selector_summary_full_df, "phase_aware_aggressive")
full_spec = lookup(selector_summary_common_subset_df, "static_speculative")
full_int8 = lookup(selector_summary_common_subset_df, "static_int8")

repeated_prefix_phase_agnostic = routing_breakdown_df[
    (routing_breakdown_df["selector"] == "phase_agnostic_simple")
    & (routing_breakdown_df["workload_name"] == "shared4032_out64")
].iloc[0]
repeated_prefix_phase_aware = routing_breakdown_df[
    (routing_breakdown_df["selector"] == "phase_aware_quality")
    & (routing_breakdown_df["workload_name"] == "shared4032_out64")
].iloc[0]

long_short_spec = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "speculative_decoding")
    & (measured_candidate_df["workload_name"] == "ctx4032_out64")
].iloc[0]
long_short_int8 = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "int8_quant")
    & (measured_candidate_df["workload_name"] == "ctx4032_out64")
].iloc[0]
long_long_spec = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "speculative_decoding")
    & (measured_candidate_df["workload_name"] == "ctx3840_out256")
].iloc[0]
long_long_int8 = measured_candidate_df[
    (measured_candidate_df["mode_name"] == "int8_quant")
    & (measured_candidate_df["workload_name"] == "ctx3840_out256")
].iloc[0]

claim_trace_df = pd.DataFrame(
    [
        {
            "claim": "On the common subset, phase_aware_quality and phase_agnostic_simple tie because they choose the same modes.",
            "value": f"{common_quality['mean_total_latency_ms']:.1f} ms mean total latency and {common_quality['mean_quality_score']:.3f} mean quality",
            "table_reference": "selector_summary_common_subset_df",
        },
        {
            "claim": "The repeated-prefix workload is where the phase-aware branch actually helps.",
            "value": (
                f"prefix_caching: {repeated_prefix_phase_agnostic['total_latency_ms']:.1f} ms, "
                f"{repeated_prefix_phase_agnostic['energy_per_token_j']:.3f} J/token; "
                f"int8_plus_prefix_caching: {repeated_prefix_phase_aware['total_latency_ms']:.1f} ms, "
                f"{repeated_prefix_phase_aware['energy_per_token_j']:.3f} J/token"
            ),
            "table_reference": "routing_breakdown_df",
        },
        {
            "claim": "The short-output long-context case is a prefill vs decode tradeoff, not just a single latency number.",
            "value": (
                f"ctx4032_out64: int8 prefill {long_short_int8['prefill_ms']:.1f} ms vs speculative {long_short_spec['prefill_ms']:.1f} ms, "
                f"but speculative tbt {long_short_spec['tbt_ms']:.3f} ms/token vs int8 {long_short_int8['tbt_ms']:.3f} ms/token"
            ),
            "table_reference": "measured_candidate_df",
        },
        {
            "claim": "The long-output case makes decode efficiency matter even more strongly.",
            "value": (
                f"ctx3840_out256: speculative tbt {long_long_spec['tbt_ms']:.3f} ms/token vs int8 {long_long_int8['tbt_ms']:.3f} ms/token"
            ),
            "table_reference": "measured_candidate_df",
        },
        {
            "claim": "The conservative controller retains enough quality relative to FP16 while the aggressive one does not.",
            "value": (
                f"phase_aware_quality retention {full_quality['quality_retention_vs_fp16']:.3f}; "
                f"phase_aware_aggressive retention {full_aggressive['quality_retention_vs_fp16']:.3f}"
            ),
            "table_reference": "selector_summary_full_df",
        },
        {
            "claim": "Among fixed baselines on the common subset, speculative is the fastest quality-safe fixed choice.",
            "value": (
                f"static_speculative total latency {full_spec['mean_total_latency_ms']:.1f} ms vs "
                f"static_int8 {full_int8['mean_total_latency_ms']:.1f} ms"
            ),
            "table_reference": "selector_summary_common_subset_df",
        },
    ]
)
claim_trace_df


,claim,value,table_reference
0,"On the common subset, phase_aware_quality and ...",1946.8 ms mean total latency and 0.918 mean qu...,selector_summary_common_subset_df
1,The repeated-prefix workload is where the phas...,"prefix_caching: 1009.5 ms, 3.821 J/token; int8...",routing_breakdown_df
2,The short-output long-context case is a prefil...,ctx4032_out64: int8 prefill 229.6 ms vs specul...,measured_candidate_df
3,The long-output case makes decode efficiency m...,ctx3840_out256: speculative tbt 7.222 ms/token...,measured_candidate_df
4,The conservative controller retains enough qua...,phase_aware_quality retention 0.891; phase_awa...,selector_summary_full_df
5,"Among fixed baselines on the common subset, sp...",static_speculative total latency 1499.5 ms vs ...,selector_summary_common_subset_df


## Serving-Oriented Extension

The serving-style row stays separate on purpose.

It is meaningful, but it is not part of the core interactive selector comparison. This section just shows what the saved measurements say for that serving-style case.


In [16]:
serving_extension_df = (
    stress_agg[stress_agg["workload_name"] == "ctx2048_out64"]
    .rename(
        columns={
            "ttft_ms_mean": "ttft_ms",
            "ttft_ms_std": "ttft_ms_std",
            "prefill_ms_mean": "prefill_ms",
            "prefill_ms_std": "prefill_ms_std",
            "tbt_ms_mean": "tbt_ms",
            "tbt_ms_std": "tbt_ms_std",
            "total_latency_ms_mean": "total_latency_ms",
            "total_latency_ms_std": "total_latency_ms_std",
            "tokens_per_second_mean": "tokens_per_second",
            "tokens_per_second_std": "tokens_per_second_std",
            "energy_per_token_j_mean": "energy_per_token_j",
            "energy_per_token_j_std": "energy_per_token_j_std",
            "peak_gpu_memory_mb_mean": "peak_gpu_memory_mb",
            "peak_gpu_memory_mb_std": "peak_gpu_memory_mb_std",
        }
    )[
        [
            "mode_name",
            "ttft_ms",
            "prefill_ms",
            "tbt_ms",
            "total_latency_ms",
            "tokens_per_second",
            "energy_per_token_j",
            "peak_gpu_memory_mb",
        ]
    ]
    .sort_values("total_latency_ms")
    .reset_index(drop=True)
)
serving_extension_df


,mode_name,ttft_ms,prefill_ms,tbt_ms,total_latency_ms,tokens_per_second,energy_per_token_j,peak_gpu_memory_mb
0,int8_plus_continuous_batching,234.742,234.742,15.143375,1203.918,212.644,1.0827,33377.31
1,gptq_plus_continuous_batching,344.721,344.721,14.610125,1279.769,200.044,1.4349,33487.54


## Final Conclusion

The clean claim that survives a strict reading is pretty narrow:

- a lightweight selector is justified
- the strongest phase-aware gain is the repeated-prefix branch
- the conservative final controller is `phase_aware_quality`
- the GPTQ-heavy route is useful as an ablation, not the final answer
- the batching hybrid result is real, but it belongs in the serving extension section rather than in the default interactive controller

The quality argument is also easier to defend this way because it is relative to the FP16 reference rather than tied to one absolute score cutoff.
